In [3]:
%pip install kagglehub


   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------- ----------------------------- 1/4 [pyyaml]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   ------------------------------ --------- 3/4 [kagglehub]
   ------------------------------ --------- 3/4 [kagglehub]
   ---------------------------------------- 4/4 [kagglehub]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os 
import kagglehub

path = kagglehub.dataset_download("sartajbhuvaji/brain-tumor-classification-mri")
print(f"Dataset downloaded to: {path}")

c:\Users\Annik\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 86.8M/86.8M [00:02<00:00, 32.7MB/s]

Extracting files...


Dataset downloaded to: C:\Users\Annik\.cache\kagglehub\datasets\sartajbhuvaji\brain-tumor-classification-mri\versions\3


In [7]:
%pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels if needed
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [9]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

train_path = os.path.join(path, "Training")
test_path = os.path.join(path, "Testing")

train_dataset = ImageFolder(root=train_path, transform=transform)
test_dataset = ImageFolder(root=test_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

classes = train_dataset.classes
print(f"Classes: {classes}")

Classes: ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']


In [10]:
import torch 
import torch.nn as nn 
from torchvision import models 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True)

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(classes))
model = model.to(device)

c:\Users\Annik\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Annik\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Annik/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 29.5MB/s]


In [11]:
import torch.optim as optim 

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [12]:
def train(model, loader, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(loader):.4f}")
    print("Training complete.")

In [13]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    class_correct = [0] * len(classes)
    class_total = [0] * len(classes)

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            for i in range(len(labels)):
                label = labels[i]
                class_total[label] += 1
                class_correct[label] += (predicted[i] == label).item()
    print(f"Overall Accuracy: {100 * correct / total:.2f}%")

    for i,c in enumerate(classes):
        acc = 100* class_correct[i] / class_total[i]
        print(f"Accuracy for class {c}: {acc:.2f}%")

In [14]:
train(model, train_loader, epochs=5)
evaluate(model, test_loader)

Epoch 1/5, Loss: 0.4671
Epoch 2/5, Loss: 0.2279
Epoch 3/5, Loss: 0.1355
Epoch 4/5, Loss: 0.0975
Epoch 5/5, Loss: 0.0764
Training complete.
Overall Accuracy: 67.01%
Accuracy for class glioma_tumor: 16.00%
Accuracy for class meningioma_tumor: 98.26%
Accuracy for class no_tumor: 100.00%
Accuracy for class pituitary_tumor: 40.54%


In [15]:
torch.save(model.state_dict(), "brain_tomor_resnet.pth")
print("Model saved as brain_tomor_resnet.pth")

Model saved as brain_tomor_resnet.pth


In [16]:
from PIL import Image 

def predict(image_path, model):
    model.eval()
    image = Image.open(image_path)
    image = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image)
        _, predicted = torch.max(output, 1)
    return classes[predicted.item()]

In [21]:
test1 = predict("test_image.jpg", model)
test2 = predict("test_image2.jpg", model)
test3 = predict("test_image3.jpg", model)
test4 = predict("test_image4.png", model)

print(f"Prediction for test_image.jpg: {test1}")
print(f"Prediction for test_image2.jpg: {test2}")
print(f"Prediction for test_image3.jpg: {test3}")
print(f"Prediction for test_image4.png: {test4}")

Prediction for test_image.jpg: meningioma_tumor
Prediction for test_image2.jpg: no_tumor
Prediction for test_image3.jpg: meningioma_tumor
Prediction for test_image4.png: meningioma_tumor
